# Off-market · real-world AVM feature shotgun

**Goal:** test hedonic-adjusted comparables, local out-of-time residuals, robust comparable-price statistics, and historical geographic tiles, **without a neural network**.

This notebook calls `avm_realworld_shotgun.py`; it does **not** reimplement training, modify the frozen off-market model, or use competition test labels.

**Fairness:** each research model is compared on identical homes to (1) a matched-budget XGBoost control and (2) the *frozen* 7.4%-MdAPE ensemble's saved August–October 2025 audit predictions. Pilot models use fewer rows/trees, so **pilot scores must not be presented as full-strength ensemble comparisons**.

Run cells top to bottom. Completed monthly predictions are reused when the configuration has not changed. A full-strength confirmation is deliberately **disabled by default**.

Full method notes: [realworld_avm_shotgun.md](docs/realworld_avm_shotgun.md).

## 1 · Locate the repository and configure the environment

Run locally from the repository root. In Google Colab, this setup optionally clones the public repository if it cannot find it. The **saved frozen-ensemble audit CSV is not tracked by Git**, so Colab users must provide a copy (e.g., mounted Drive).

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Optional configuration: specify your own checkout path or historical data folder.
PROJECT_ROOT = None                       # e.g. r"C:\path\to\SSU-Data-Dash-hdnl-comp"
DATA_DIR_OVERRIDE = None                  # None keeps your existing DATA_DIR setting
CLONE_IN_COLAB_IF_MISSING = True
INSTALL_REQUIREMENTS_IN_COLAB = True

def locate_project():
    if PROJECT_ROOT is not None:
        root = Path(PROJECT_ROOT).expanduser().resolve()
        if not (root / "avm_realworld_shotgun.py").is_file():
            raise FileNotFoundError(f"Not a project checkout: {root}")
        return root
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        for candidate in (parent, parent / "SSU-Data-Dash-hdnl-comp"):
            if (candidate / "avm_realworld_shotgun.py").is_file():
                return candidate
    if "google.colab" in sys.modules and CLONE_IN_COLAB_IF_MISSING:
        target = Path("/content/SSU-Data-Dash-hdnl-comp")
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
             str(target)],
            check=True,
        )
        return target
    raise FileNotFoundError("Open this notebook from the repository, or set PROJECT_ROOT.")

ROOT = locate_project()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if DATA_DIR_OVERRIDE:
    os.environ["DATA_DIR"] = str(Path(DATA_DIR_OVERRIDE).expanduser().resolve())
if "google.colab" in sys.modules and INSTALL_REQUIREMENTS_IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(ROOT / "requirements.txt")], check=True)
print("Project:", ROOT)
print("Python:", sys.version.split()[0])
print("DATA_DIR:", os.environ.get("DATA_DIR", "(project data/ or Hugging Face)"))

## 2 · Choose the shotgun budget and feature families

Defaults mirror the CLI's three-month pilot: 10,000 historical properties, 220 trees and 350 held-out homes per month. Change `OUTPUT_DIR` when changing a configuration—cached output folders enforce a reproducibility manifest.

The first arm is always the **matched-budget XGBoost baseline**. Additional control arms correspond to earlier feature-engineering experiments; the `rw_*` arms are new.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from avm_realworld_shotgun import (
    DEFAULT_FEATURES, DEFAULT_REFERENCE, run_realworld_shotgun,
)
from models.spacetime_screen_features import FEATURE_VARIANTS

MONTHS = ("2025-08", "2025-09", "2025-10")
FEATURES = tuple(DEFAULT_FEATURES)
# To focus first: ("baseline", "rw_adjusted_comps", "rw_local_residuals", "rw_combined")
REFERENCE_FILE = Path(DEFAULT_REFERENCE)
OUTPUT_DIR = Path("outputs/realworld_avm_pilot")

RUN_TESTS = False
RUN_PILOT = True
PILOT_TRAIN_CAP = 10_000
PILOT_VALIDATION_CAP = 350
PILOT_TREES = 220
BOOTSTRAP_REPEATS = 300
SEED = 42

assert "baseline" in FEATURES and len(set(FEATURES)) == len(FEATURES)
assert set(FEATURES).issubset(set(FEATURE_VARIANTS))
display(pd.DataFrame({"feature_arm": list(FEATURES)}))
print("Frozen reference:", REFERENCE_FILE)
print("Pilot outputs:", OUTPUT_DIR)

## 3 · Preconditions

The **frozen historical reference** must already exist as `outputs/ensemble_release_candidate_full/shadow_audit_predictions.csv`. This file is generated by the earlier ensemble audit—not by `finalize` or `predict`.

If running on Colab, set `REFERENCE_FILE` to an existing Drive CSV, or copy the saved CSV into the matching `outputs/` path. This notebook won't retrain the ensemble just to reconstruct its reference predictions.

In [ ]:
import data
import numpy as np

# Set True in a separate run if you want the synthetic regression tests first.
if RUN_TESTS:
    subprocess.run(
        [sys.executable, "-m", "pytest",
         "tests/test_realworld_avm_shotgun.py", "-q"],
        check=True, cwd=ROOT,
    )

if not REFERENCE_FILE.is_file():
    raise FileNotFoundError(
        f"Missing frozen ensemble reference: {REFERENCE_FILE.resolve()}\n"
        "Copy the historical shadow_audit_predictions.csv from your previous "
        "AutoML run or set REFERENCE_FILE to its actual path. "
        "Do NOT substitute hidden-test predictions."
    )
historical_path = data.find_file("listings")
ref = pd.read_csv(REFERENCE_FILE, dtype={"ml_num": str})
required = {"month", "ml_num", "sold_price", "predicted_price"}
assert required <= set(ref), f"Reference missing: {required-set(ref)}"
assert not ref.duplicated(["month", "ml_num"]).any(), "Duplicate reference homes"
print("Historical sales:", historical_path)
print("Reference homes:", len(ref))
print("Reference months:", sorted(ref.month.astype(str).unique()))
print("Research months:", MONTHS)
display(ref.head(3))

## 4 · Run or resume the pilot

Only the new research models are fitted. The frozen ensemble is read from its CSV. The pilot checkpoints each completed month and can be safely resumed with **unchanged settings, input data and reference CSV**.

In [ ]:
if RUN_PILOT:
    pooled, monthly, paired = run_realworld_shotgun(
        months=MONTHS,
        features=FEATURES,
        full=False,
        train_cap=PILOT_TRAIN_CAP,
        validation_cap=PILOT_VALIDATION_CAP,
        trees=PILOT_TREES,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        seed=SEED,
        reference_file=REFERENCE_FILE,
        output_dir=OUTPUT_DIR,
    )
else:
    # Inspect an earlier completed run without fitting anything.
    pooled = pd.read_csv(OUTPUT_DIR / "pooled_vs_ensemble.csv")
    monthly = pd.read_csv(OUTPUT_DIR / "monthly_vs_ensemble.csv")
    paired = pd.read_csv(OUTPUT_DIR / "paired_predictions.csv",
                         dtype={"ml_num": str})
print("Loaded", len(paired), "model/home rows.")

## 5 · Matched-home leaderboard

**Primary pilot selection criterion:** `delta_control_mdape_pp` — a negative value means the feature-enhanced XGBoost improved on the **same 220-tree XGBoost baseline**, with the same training/validation homes. The 700-tree ensemble's `delta_ensemble_mdape_pp` is contextual and **not** a matched-capacity comparison.

In [ ]:
show = [
    "arm", "variant", "homes", "mdape", "mape",
    "delta_control_mdape_pp", "delta_control_mape_pp",
    "delta_ensemble_mdape_pp", "delta_ensemble_mape_pp",
    "fraction_better_than_control", "fraction_better_than_ensemble",
]
display(pooled[[c for c in show if c in pooled]].round(4).reset_index(drop=True))
reference = pooled.loc[
    (pooled.arm == "frozen_reference") & (pooled.variant == "ensemble")
]
control = pooled.loc[
    (pooled.arm == "feature") & (pooled.variant == "baseline")
]
assert len(reference) == len(control) == 1
print(f"Frozen ensemble on THIS subset: MdAPE {reference.mdape.iloc[0]:.4f}%, "
      f"MAPE {reference.mape.iloc[0]:.4f}%")
print(f"Matched-budget XGBoost control: MdAPE {control.mdape.iloc[0]:.4f}%, "
      f"MAPE {control.mape.iloc[0]:.4f}%")

In [ ]:
import matplotlib.pyplot as plt

candidates = pooled.loc[
    (pooled.arm == "feature") & (pooled.variant != "baseline")
].sort_values("delta_control_mdape_pp")
if len(candidates):
    fig, ax = plt.subplots(figsize=(9, max(3, 0.45 * len(candidates))))
    ax.barh(candidates.variant, candidates.delta_control_mdape_pp)
    ax.axvline(0, color="gray", linewidth=1)
    ax.invert_yaxis()
    ax.set_xlabel("Change in MdAPE vs matched-budget XGBoost (percentage points)")
    ax.set_title("Off-market feature ablation — lower is better")
    plt.tight_layout()
    plt.show()

## 6 · Month-by-month stability and paired-home diagnostics

A pooled improvement is more convincing if it persists across months rather than reflecting one lucky sample. Individual error comparisons identify whether gains are concentrated in a small subset of properties.

In [ ]:
monthly_control = monthly.loc[monthly.arm == "feature"]
display(monthly_control.pivot_table(
    index="variant", columns="month", values="delta_ensemble_mdape_pp"
).round(3))
print("Month deltas above compare with the frozen ensemble on each month's same subset.")
print("For feature selection, use pooled 'delta_control_mdape_pp' first.")

# Derived per-property APE differences for one candidate vs matched XGBoost.
feature_rows = pooled.loc[
    (pooled.arm == "feature") & (pooled.variant != "baseline")
].sort_values("delta_control_mdape_pp")
if len(feature_rows):
    best_name = feature_rows.variant.iloc[0]
    keys = ["month", "ml_num"]
    base = paired.loc[(paired.arm == "feature") & (paired.variant == "baseline")].set_index(keys)
    best = paired.loc[(paired.arm == "feature") & (paired.variant == best_name)].set_index(keys)
    joined = base[["sold_price", "predicted_price"]].join(
        best[["predicted_price"]], lsuffix="_base", rsuffix="_challenger",
        validate="one_to_one",
    )
    joined["base_ape"] = 100 * abs(joined.predicted_price_base - joined.sold_price) / joined.sold_price
    joined["challenger_ape"] = 100 * abs(joined.predicted_price_challenger - joined.sold_price) / joined.sold_price
    joined["ape_improvement_pp"] = joined.base_ape - joined.challenger_ape
    print("Best pilot candidate:", best_name)
    display(joined.sort_values("ape_improvement_pp", ascending=False).head(10).round(3))
    display(joined.sort_values("ape_improvement_pp", ascending=True).head(10).round(3))

## 7 · Optional full-strength confirmation (off by default)

After reviewing the pilot, edit the shortlist and change `RUN_FULL = True`. Full mode retrains challengers with 700 trees and all eligible historical sales, evaluating the complete audit cohorts. It still does **not** refit or alter the frozen ensemble.

**Do not blindly promote a full-mode winner on this previously inspected audit period.** Use it as development evidence, then freeze any proposed change before another chronological diagnostic.

In [ ]:
RUN_FULL = False
FULL_FEATURES = ("baseline", "rw_adjusted_comps", "rw_combined")
FULL_OUTPUT_DIR = Path("outputs/realworld_avm_full")

if RUN_FULL:
    full_pooled, full_monthly, full_paired = run_realworld_shotgun(
        months=MONTHS,
        features=FULL_FEATURES,
        full=True,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        seed=SEED,
        reference_file=REFERENCE_FILE,
        output_dir=FULL_OUTPUT_DIR,
    )
    display(full_pooled.round(4))
else:
    print("Full-strength off-market training is disabled; set RUN_FULL=True to opt in.")

## 8 · Interpretation and next steps

A feature is worth promoting when it improves **matched-budget MdAPE**, retains acceptable MAPE/P90 behaviour, and shows credible month-to-month robustness. Note that this experiment compares individual XGBoost challengers to the weighted ensemble—it does not measure whether adding them as new ensemble members improves the ensemble.

After reviewing the CSVs, continue to the **[list-aware showdown notebook](listaware_shotgun.ipynb)** for CatBoost Huber + XGBoost spline/geography features, adjusted comparables, LightGBM, and fixed-weight cross-family blends.

All actual scores must come from saved evaluation CSVs; never infer a new score from the published 7.4% historical reference.